# Workshop Databricks - Telefonica Onboarding

## Bem-vindo à Plataforma Databricks! 🚀

Este workshop foi desenvolvido para apresentar as principais funcionalidades da plataforma Databricks, com foco em:

* **Unity Catalog**: Governança de dados unificada e gerenciamento de metadados
* **Delta Lake**: Formato de armazenamento com recursos avançados como Time Travel
* **Metric Views**: Camada semântica para análises consistentes e reutilizáveis
* **Dashboards & Genie Spaces**: Visualização e análise de dados com IA

---

### Agenda do Workshop

1. **Configuração Inicial** - Criação de catálogo e schema no Unity Catalog
2. **Dados de Demonstração** - Geração de dataset simulado de telecomunicações
3. **Unity Catalog em Ação** - Exploração de recursos de governança
4. **Delta Lake Time Travel** - Versionamento e consultas históricas
5. **Metric Views** - Definição de métricas de negócio reutilizáveis
6. **Dashboard & Análise Semântica** - Criação de visualizações e Genie Space

## 1. Configuração Inicial - Unity Catalog

O **Unity Catalog** é a solução de governança de dados da Databricks que fornece:

* Gerenciamento centralizado de metadados
* Controle de acesso granular (tabelas, colunas, linhas)
* Auditoria e linhagem de dados
* Compartilhamento seguro de dados entre organizações

### Estrutura Hierárquica

```
Catalog (Catálogo)
  └── Schema (Esquema/Database)
      └── Tables, Views, Functions
```

Vamos criar nossa estrutura para o workshop:

In [0]:
# Configuração do ambiente
import random
import string
from datetime import datetime, timedelta
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Gerar um identificador único para evitar conflitos
user_id = spark.sql("SELECT current_user()").collect()[0][0].split('@')[0].replace('.', '_')
timestamp_id = datetime.now().strftime("%Y%m%d")

# Nomes do catálogo e schema
catalog_name = f"workshop_telefonica_{user_id}"
schema_name = "telecom_data"

print(f"📦 Catálogo: {catalog_name}")
print(f"📂 Schema: {schema_name}")
print(f"\n🔧 Criando estrutura no Unity Catalog...")

# Criar catálogo (se não existir)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"USE CATALOG {catalog_name}")

# Criar schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name} COMMENT 'Dados de telecomunicações para workshop'")
spark.sql(f"USE SCHEMA {schema_name}")

print(f"\n✅ Estrutura criada com sucesso!")
print(f"\n💡 Você pode explorar no Data Explorer: {catalog_name}.{schema_name}")

📦 Catálogo: workshop_telefonica_reinaldo_costa
📂 Schema: telecom_data

🔧 Criando estrutura no Unity Catalog...

✅ Estrutura criada com sucesso!

💡 Você pode explorar no Data Explorer: workshop_telefonica_reinaldo_costa.telecom_data


## 2. Geração de Dados de Demonstração

Vamos criar um dataset simulado de telecomunicações com:

* **Clientes**: Informações de assinantes
* **Planos**: Tipos de planos oferecidos
* **Uso de Dados**: Registros de consumo diário
* **Receita**: Transações e faturamento

Estes dados serão armazenados como **Delta Tables** no Unity Catalog.

In [0]:
# Gerar tabela de clientes
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType

print("👥 Gerando dados de clientes...")

# Dados de clientes
clientes_data = []
for i in range(1, 1001):
    cliente_id = f"CLI{i:05d}"
    nome = f"Cliente {i}"
    segmento = random.choice(["Residencial", "Empresarial", "Premium"])
    cidade = random.choice(["São Paulo", "Rio de Janeiro", "Brasília", "Salvador", "Belo Horizonte"])
    data_cadastro = (datetime.now() - timedelta(days=random.randint(30, 1095))).date()
    
    clientes_data.append((cliente_id, nome, segmento, cidade, data_cadastro))

schema_clientes = StructType([
    StructField("cliente_id", StringType(), False),
    StructField("nome", StringType(), False),
    StructField("segmento", StringType(), False),
    StructField("cidade", StringType(), False),
    StructField("data_cadastro", DateType(), False)
])

df_clientes = spark.createDataFrame(clientes_data, schema_clientes)

# Salvar como Delta Table
df_clientes.write.format("delta").mode("overwrite").saveAsTable("clientes")

print(f"✅ Tabela 'clientes' criada com {df_clientes.count()} registros")
display(df_clientes.limit(5))

👥 Gerando dados de clientes...
✅ Tabela 'clientes' criada com 1000 registros


cliente_id,nome,segmento,cidade,data_cadastro
CLI00001,Cliente 1,Premium,São Paulo,2023-08-02
CLI00002,Cliente 2,Residencial,Rio de Janeiro,2023-11-17
CLI00003,Cliente 3,Empresarial,Salvador,2025-02-07
CLI00004,Cliente 4,Empresarial,Brasília,2024-07-22
CLI00005,Cliente 5,Premium,São Paulo,2023-08-30


In [0]:
# Gerar tabela de planos
print("📊 Gerando dados de planos...")

planos_data = [
    ("PLAN001", "Básico 5GB", "Pré-pago", 5, 29.90),
    ("PLAN002", "Intermediário 15GB", "Pós-pago", 15, 59.90),
    ("PLAN003", "Avançado 30GB", "Pós-pago", 30, 89.90),
    ("PLAN004", "Premium 50GB", "Pós-pago", 50, 129.90),
    ("PLAN005", "Ilimitado", "Pós-pago", 999, 199.90),
    ("PLAN006", "Empresarial 100GB", "Corporativo", 100, 299.90),
    ("PLAN007", "Empresarial Ilimitado", "Corporativo", 999, 499.90)
]

schema_planos = StructType([
    StructField("plano_id", StringType(), False),
    StructField("nome_plano", StringType(), False),
    StructField("tipo", StringType(), False),
    StructField("franquia_gb", IntegerType(), False),
    StructField("valor_mensal", DoubleType(), False)
])

df_planos = spark.createDataFrame(planos_data, schema_planos)
df_planos.write.format("delta").mode("overwrite").saveAsTable("planos")

print(f"✅ Tabela 'planos' criada com {df_planos.count()} registros")
display(df_planos)

📊 Gerando dados de planos...
✅ Tabela 'planos' criada com 7 registros


plano_id,nome_plano,tipo,franquia_gb,valor_mensal
PLAN001,Básico 5GB,Pré-pago,5,29.9
PLAN002,Intermediário 15GB,Pós-pago,15,59.9
PLAN003,Avançado 30GB,Pós-pago,30,89.9
PLAN004,Premium 50GB,Pós-pago,50,129.9
PLAN005,Ilimitado,Pós-pago,999,199.9
PLAN006,Empresarial 100GB,Corporativo,100,299.9
PLAN007,Empresarial Ilimitado,Corporativo,999,499.9


In [0]:
# Gerar dados de uso diário (90 dias)
print("📱 Gerando dados de uso de dados...")

uso_data = []
data_inicio = datetime.now() - timedelta(days=90)

# Selecionar amostra de clientes para gerar uso
clientes_sample = [f"CLI{i:05d}" for i in range(1, 501)]  # 500 clientes
planos_ids = ["PLAN001", "PLAN002", "PLAN003", "PLAN004", "PLAN005", "PLAN006", "PLAN007"]

# Use Python's built-in round (not PySpark's round)
import builtins

for cliente_id in clientes_sample:
    plano_id = random.choice(planos_ids)
    
    # Gerar 90 dias de uso
    for dia in range(90):
        data_uso = (data_inicio + timedelta(days=dia)).date()
        
        # Consumo varia por tipo de plano
        if "Ilimitado" in plano_id or plano_id in ["PLAN005", "PLAN007"]:
            consumo_gb = builtins.round(random.uniform(5, 25), 2)
        else:
            consumo_gb = builtins.round(random.uniform(0.5, 8), 2)
        
        minutos_voz = random.randint(10, 300)
        sms_enviados = random.randint(0, 50)
        
        uso_data.append((cliente_id, plano_id, data_uso, consumo_gb, minutos_voz, sms_enviados))

schema_uso = StructType([
    StructField("cliente_id", StringType(), False),
    StructField("plano_id", StringType(), False),
    StructField("data_uso", DateType(), False),
    StructField("consumo_gb", DoubleType(), False),
    StructField("minutos_voz", IntegerType(), False),
    StructField("sms_enviados", IntegerType(), False)
])

df_uso = spark.createDataFrame(uso_data, schema_uso)
df_uso.write.format("delta").mode("overwrite").saveAsTable("uso_diario")

print(f"✅ Tabela 'uso_diario' criada com {df_uso.count():,} registros")
print(f"\n📅 Período: {data_inicio.date()} até {datetime.now().date()}")
display(df_uso.limit(10))

📱 Gerando dados de uso de dados...
✅ Tabela 'uso_diario' criada com 45,000 registros

📅 Período: 2025-09-16 até 2025-12-15


cliente_id,plano_id,data_uso,consumo_gb,minutos_voz,sms_enviados
CLI00001,PLAN007,2025-09-16,12.65,183,3
CLI00001,PLAN007,2025-09-17,18.13,237,35
CLI00001,PLAN007,2025-09-18,9.14,223,2
CLI00001,PLAN007,2025-09-19,9.07,252,13
CLI00001,PLAN007,2025-09-20,18.24,227,8
CLI00001,PLAN007,2025-09-21,24.67,90,20
CLI00001,PLAN007,2025-09-22,20.07,49,47
CLI00001,PLAN007,2025-09-23,14.39,104,4
CLI00001,PLAN007,2025-09-24,23.72,93,43
CLI00001,PLAN007,2025-09-25,24.08,282,6


In [0]:
# Gerar dados de receita mensal
print("💰 Gerando dados de receita...")

# Use Python's built-in round (not PySpark's round)
import builtins

receita_data = []

# Gerar receita para os últimos 3 meses
for mes_offset in range(3):
    data_ref = datetime.now() - timedelta(days=30 * mes_offset)
    mes_ano = data_ref.strftime("%Y-%m")
    
    for cliente_id in clientes_sample:
        plano_id = random.choice(planos_ids)
        
        # Buscar valor do plano
        valor_base = next((p[4] for p in planos_data if p[0] == plano_id), 50.0)
        
        # Adicionar variações (excedentes, descontos, etc)
        valor_excedente = builtins.round(random.uniform(0, 30), 2) if random.random() > 0.7 else 0
        desconto = builtins.round(random.uniform(0, 20), 2) if random.random() > 0.8 else 0
        
        valor_total = builtins.round(valor_base + valor_excedente - desconto, 2)
        status_pagamento = random.choice(["Pago", "Pago", "Pago", "Pendente", "Atrasado"])
        
        data_vencimento = data_ref.replace(day=10).date()
        data_pagamento = (data_ref + timedelta(days=random.randint(-5, 15))).date() if status_pagamento == "Pago" else None
        
        receita_data.append((cliente_id, plano_id, mes_ano, data_vencimento, valor_base, 
                           valor_excedente, desconto, valor_total, status_pagamento, data_pagamento))

schema_receita = StructType([
    StructField("cliente_id", StringType(), False),
    StructField("plano_id", StringType(), False),
    StructField("mes_referencia", StringType(), False),
    StructField("data_vencimento", DateType(), False),
    StructField("valor_base", DoubleType(), False),
    StructField("valor_excedente", DoubleType(), False),
    StructField("desconto", DoubleType(), False),
    StructField("valor_total", DoubleType(), False),
    StructField("status_pagamento", StringType(), False),
    StructField("data_pagamento", DateType(), True)
])

df_receita = spark.createDataFrame(receita_data, schema_receita)
df_receita.write.format("delta").mode("overwrite").saveAsTable("receita_mensal")

print(f"✅ Tabela 'receita_mensal' criada com {df_receita.count():,} registros")
display(df_receita.limit(10))

💰 Gerando dados de receita...
✅ Tabela 'receita_mensal' criada com 1,500 registros


cliente_id,plano_id,mes_referencia,data_vencimento,valor_base,valor_excedente,desconto,valor_total,status_pagamento,data_pagamento
CLI00001,PLAN002,2025-12,2025-12-10,59.9,0.0,5.32,54.58,Pago,2025-12-14
CLI00002,PLAN006,2025-12,2025-12-10,299.9,13.96,0.0,313.86,Pendente,null
CLI00003,PLAN003,2025-12,2025-12-10,89.9,0.0,0.0,89.9,Atrasado,null
CLI00004,PLAN004,2025-12,2025-12-10,129.9,0.0,0.0,129.9,Pendente,null
CLI00005,PLAN006,2025-12,2025-12-10,299.9,0.0,12.89,287.01,Pendente,null
CLI00006,PLAN006,2025-12,2025-12-10,299.9,3.6,0.0,303.5,Pago,2025-12-26
CLI00007,PLAN002,2025-12,2025-12-10,59.9,0.0,0.0,59.9,Atrasado,null
CLI00008,PLAN007,2025-12,2025-12-10,499.9,23.99,0.0,523.89,Pago,2025-12-16
CLI00009,PLAN003,2025-12,2025-12-10,89.9,0.0,5.02,84.88,Atrasado,null
CLI00010,PLAN006,2025-12,2025-12-10,299.9,0.0,0.0,299.9,Pendente,null


In [0]:
# Resumo das tabelas criadas
print("\n" + "="*60)
print("🎉 DADOS CRIADOS COM SUCESSO!")
print("="*60)

tabelas = spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}").collect()

print(f"\n📁 Catálogo: {catalog_name}")
print(f"📂 Schema: {schema_name}")
print(f"\n📊 Tabelas criadas:\n")

for tabela in tabelas:
    nome_tabela = tabela.tableName
    count = spark.table(f"{catalog_name}.{schema_name}.{nome_tabela}").count()
    print(f"  • {nome_tabela}: {count:,} registros")

print(f"\n🔗 Acesse no Data Explorer: {catalog_name}.{schema_name}")
print("\n➡️ Próximos passos: Explorar recursos do Unity Catalog e Delta Lake!")


🎉 DADOS CRIADOS COM SUCESSO!

📁 Catálogo: workshop_telefonica_reinaldo_costa
📂 Schema: telecom_data

📊 Tabelas criadas:

  • clientes: 1,000 registros
  • planos: 7 registros
  • receita_mensal: 1,500 registros
  • uso_diario: 45,000 registros

🔗 Acesse no Data Explorer: workshop_telefonica_reinaldo_costa.telecom_data

➡️ Próximos passos: Explorar recursos do Unity Catalog e Delta Lake!


## 3. Unity Catalog em Ação

Vamos explorar os recursos de governança e gerenciamento de metadados do Unity Catalog:

### Recursos que vamos demonstrar:

* **Metadados Enriquecidos**: Comentários e documentação de tabelas e colunas
* **Propriedades de Tabelas**: Informações sobre formato, localização e estatísticas
* **Linhagem de Dados**: Rastreamento de origem e uso dos dados
* **Controle de Acesso**: Permissões granulares (demonstração conceitual)
* **Auditoria**: Histórico de operações

In [0]:
%sql
-- Adicionar comentários às tabelas para documentação
COMMENT ON TABLE clientes IS 'Cadastro de clientes da Telefonica - contém informações de assinantes e segmentação';
COMMENT ON TABLE planos IS 'Catálogo de planos oferecidos - inclui franquias e preços';
COMMENT ON TABLE uso_diario IS 'Registros diários de consumo de dados, voz e SMS por cliente';
COMMENT ON TABLE receita_mensal IS 'Faturamento mensal por cliente - inclui valores base, excedentes e status de pagamento';

-- Adicionar comentários em colunas específicas
ALTER TABLE clientes ALTER COLUMN cliente_id COMMENT 'Identificador único do cliente (formato: CLI00000)';
ALTER TABLE clientes ALTER COLUMN segmento COMMENT 'Segmentação: Residencial, Empresarial ou Premium';

ALTER TABLE uso_diario ALTER COLUMN consumo_gb COMMENT 'Consumo de dados em Gigabytes';
ALTER TABLE uso_diario ALTER COLUMN minutos_voz COMMENT 'Minutos de ligações de voz';

SELECT '✅ Comentários adicionados com sucesso!' as status

status
✅ Comentários adicionados com sucesso!


In [0]:
%sql
-- Visualizar metadados detalhados de uma tabela
DESCRIBE EXTENDED clientes

col_name,data_type,comment
cliente_id,string,Identificador único do cliente (formato: CLI00000)
nome,string,null
segmento,string,"Segmentação: Residencial, Empresarial ou Premium"
cidade,string,null
data_cadastro,date,null
,,
# Delta Statistics Columns,,
Column Names,"segmento, cidade, cliente_id, nome, data_cadastro",
Column Selection Method,first-32,
,,


In [0]:
# Explorar propriedades das tabelas Delta
print("🔍 Explorando propriedades das tabelas Delta\n")

tabelas = ["clientes", "planos", "uso_diario", "receita_mensal"]

for tabela in tabelas:
    print(f"\n{'='*60}")
    print(f"📊 Tabela: {tabela}")
    print(f"{'='*60}")
    
    # Obter detalhes da tabela
    detalhes = spark.sql(f"DESCRIBE DETAIL {catalog_name}.{schema_name}.{tabela}").collect()[0]
    
    print(f"  • Formato: {detalhes.format}")
    print(f"  • Localização: {detalhes.location}")
    print(f"  • Número de arquivos: {detalhes.numFiles}")
    print(f"  • Tamanho: {detalhes.sizeInBytes / 1024 / 1024:.2f} MB")
    print(f"  • Última modificação: {detalhes.lastModified}")
    
print(f"\n\n💡 Todas as tabelas estão no formato Delta Lake!")

🔍 Explorando propriedades das tabelas Delta


📊 Tabela: clientes
  • Formato: delta
  • Localização: s3://databricks-e2demofieldengwest/b169b504-4c54-49f2-bc3a-adf4b128f36d/tables/27c4198c-73ed-4250-992a-52f1e1deca6e
  • Número de arquivos: 1
  • Tamanho: 0.01 MB
  • Última modificação: 2025-12-15 23:33:01

📊 Tabela: planos
  • Formato: delta
  • Localização: s3://databricks-e2demofieldengwest/b169b504-4c54-49f2-bc3a-adf4b128f36d/tables/97be2e4d-1420-49c8-9a1c-0fc9499fd9e3
  • Número de arquivos: 1
  • Tamanho: 0.00 MB
  • Última modificação: 2025-12-15 23:32:58

📊 Tabela: uso_diario
  • Formato: delta
  • Localização: s3://databricks-e2demofieldengwest/b169b504-4c54-49f2-bc3a-adf4b128f36d/tables/2710b8ef-199c-482d-85e5-4693a31cc7fe
  • Número de arquivos: 32
  • Tamanho: 0.31 MB
  • Última modificação: 2025-12-15 23:33:03

📊 Tabela: receita_mensal
  • Formato: delta
  • Localização: s3://databricks-e2demofieldengwest/b169b504-4c54-49f2-bc3a-adf4b128f36d/tables/e2e737bc-77a5-45a2-b6b1-

In [0]:
%sql
-- Consultar o Information Schema do Unity Catalog
-- Listar todas as tabelas com seus comentários

SELECT 
  table_name,
  table_type,
  comment,
  created
FROM system.information_schema.tables
WHERE table_catalog = current_catalog()
  AND table_schema = current_schema()
ORDER BY table_name

table_name,table_type,comment,created
clientes,MANAGED,Cadastro de clientes da Telefonica - contém informações de assinantes e segmentação,2025-12-15T23:28:08.845Z
planos,MANAGED,Catálogo de planos oferecidos - inclui franquias e preços,2025-12-15T23:28:28.807Z
receita_mensal,MANAGED,"Faturamento mensal por cliente - inclui valores base, excedentes e status de pagamento",2025-12-15T23:32:06.322Z
uso_diario,MANAGED,"Registros diários de consumo de dados, voz e SMS por cliente",2025-12-15T23:31:04.107Z


In [0]:
%sql
-- Visualizar metadados de colunas com comentários

SELECT 
  table_name,
  column_name,
  data_type,
  comment
FROM system.information_schema.columns
WHERE table_catalog = current_catalog()
  AND table_schema = current_schema()
  AND table_name = 'clientes'
ORDER BY ordinal_position

table_name,column_name,data_type,comment
clientes,cliente_id,STRING,Identificador único do cliente (formato: CLI00000)
clientes,nome,STRING,null
clientes,segmento,STRING,"Segmentação: Residencial, Empresarial ou Premium"
clientes,cidade,STRING,null
clientes,data_cadastro,DATE,null


### 🎯 Benefícios do Unity Catalog Demonstrados

✅ **Metadados Centralizados**: Todos os comentários e documentações em um só lugar

✅ **Descoberta de Dados**: Fácil localização de tabelas e colunas através do Data Explorer

✅ **Governança**: Controle granular de acesso (nível de catálogo, schema, tabela, coluna)

✅ **Auditoria**: Rastreamento completo de quem acessa e modifica os dados

✅ **Linhagem**: Visualização de origem e destino dos dados (disponível no Data Explorer)

---

💡 **Dica**: No Data Explorer, você pode visualizar graficamente a linhagem de dados, permissões e histórico de acesso!

## 4. Delta Lake Time Travel

O **Delta Lake** é um formato de armazenamento open-source que traz confiabilidade para data lakes:

### Principais Recursos:

* **ACID Transactions**: Garantia de consistência dos dados
* **Time Travel**: Acesso a versões históricas dos dados
* **Schema Evolution**: Evolução do schema sem quebrar pipelines
* **Audit History**: Histórico completo de todas as operações
* **Upserts e Deletes**: Operações DML eficientes

Vamos demonstrar o **Time Travel** fazendo modificações na tabela de clientes!

In [0]:
%sql
-- Visualizar histórico de versões da tabela
DESCRIBE HISTORY clientes

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2025-12-15T23:33:01.000Z,3966843094035027,reinaldo.costa@databricks.com,CHANGE COLUMN,"Map(column -> {""name"":""segmento"",""type"":""string"",""nullable"":true,""metadata"":{""comment"":""Segmentação: Residencial, Empresarial ou Premium""}})",null,List(2945663160424627),1215-132937-hmm5sui5-v2n,2,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-photon-scala2.13
2,2025-12-15T23:33:00.000Z,3966843094035027,reinaldo.costa@databricks.com,CHANGE COLUMN,"Map(column -> {""name"":""cliente_id"",""type"":""string"",""nullable"":true,""metadata"":{""comment"":""Identificador único do cliente (formato: CLI00000)""}})",null,List(2945663160424627),1215-132937-hmm5sui5-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-photon-scala2.13
1,2025-12-15T23:32:57.000Z,3966843094035027,reinaldo.costa@databricks.com,SET TBLPROPERTIES,"Map(properties -> {""comment"":""Cadastro de clientes da Telefonica - contém informações de assinantes e segmentação""})",null,List(2945663160424627),1215-132937-hmm5sui5-v2n,0,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-photon-scala2.13
0,2025-12-15T23:28:09.000Z,3966843094035027,reinaldo.costa@databricks.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(2945663160424627),1215-132937-hmm5sui5-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000, numOutputBytes -> 6119)",null,Databricks-Runtime/17.3.x-photon-scala2.13


In [0]:
# Simular adição de novos clientes
print("🔄 Atualização 1: Adicionando novos clientes...\n")

novos_clientes = [
    ("CLI01001", "Maria Silva", "Premium", "São Paulo", datetime.now().date()),
    ("CLI01002", "João Santos", "Empresarial", "Rio de Janeiro", datetime.now().date()),
    ("CLI01003", "Ana Costa", "Residencial", "Brasília", datetime.now().date())
]

df_novos = spark.createDataFrame(novos_clientes, schema_clientes)
df_novos.write.format("delta").mode("append").saveAsTable("clientes")

count_atual = spark.table("clientes").count()
print(f"✅ 3 novos clientes adicionados!")
print(f"📊 Total de clientes agora: {count_atual}")

# Mostrar os novos clientes
print("\n🆕 Novos clientes:")
display(spark.sql("SELECT * FROM clientes WHERE cliente_id >= 'CLI01001' ORDER BY cliente_id"))

🔄 Atualização 1: Adicionando novos clientes...

✅ 3 novos clientes adicionados!
📊 Total de clientes agora: 1003

🆕 Novos clientes:


cliente_id,nome,segmento,cidade,data_cadastro
CLI01001,Maria Silva,Premium,São Paulo,2025-12-16
CLI01002,João Santos,Empresarial,Rio de Janeiro,2025-12-16
CLI01003,Ana Costa,Residencial,Brasília,2025-12-16


In [0]:
%sql
-- Atualizar segmento de alguns clientes
UPDATE clientes 
SET segmento = 'Premium'
WHERE cliente_id IN ('CLI00001', 'CLI00002', 'CLI00003')
  AND segmento != 'Premium';

SELECT '3 clientes atualizados para segmento Premium' as status

status
3 clientes atualizados para segmento Premium


In [0]:
%sql
-- Deletar um cliente de teste
DELETE FROM clientes WHERE cliente_id = 'CLI01003';

SELECT 'Cliente CLI01003 removido' as status

status
Cliente CLI01003 removido


In [0]:
%sql
-- Visualizar histórico atualizado com todas as operações
SELECT 
  version,
  timestamp,
  operation,
  operationParameters,
  operationMetrics
FROM (DESCRIBE HISTORY clientes)
ORDER BY version DESC
LIMIT 10

version,timestamp,operation,operationParameters,operationMetrics
7,2025-12-16T00:29:16.000Z,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)","Map(numRemovedFiles -> 3, numRemovedBytes -> 9526, p25FileSize -> 6405, numDeletionVectorsRemoved -> 2, minFileSize -> 6405, numAddedFiles -> 1, maxFileSize -> 6405, p75FileSize -> 6405, p50FileSize -> 6405, numAddedBytes -> 6405)"
6,2025-12-16T00:29:15.000Z,DELETE,"Map(predicate -> [""(cliente_id#3634636 = CLI01003)""])","Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 656, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 482, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 173)"
5,2025-12-16T00:28:59.000Z,UPDATE,"Map(predicate -> [""(cliente_id#3632055 IN (CLI00001,CLI00002,CLI00003) AND NOT (segmento#3632057 = Premium))""])","Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1282, numDeletionVectorsUpdated -> 0, scanTimeMs -> 667, numAddedFiles -> 1, numUpdatedRows -> 2, numAddedBytes -> 1739, rewriteTimeMs -> 611)"
4,2025-12-16T00:28:36.000Z,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])","Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1668)"
3,2025-12-15T23:33:01.000Z,CHANGE COLUMN,"Map(column -> {""name"":""segmento"",""type"":""string"",""nullable"":true,""metadata"":{""comment"":""Segmentação: Residencial, Empresarial ou Premium""}})",Map()
2,2025-12-15T23:33:00.000Z,CHANGE COLUMN,"Map(column -> {""name"":""cliente_id"",""type"":""string"",""nullable"":true,""metadata"":{""comment"":""Identificador único do cliente (formato: CLI00000)""}})",Map()
1,2025-12-15T23:32:57.000Z,SET TBLPROPERTIES,"Map(properties -> {""comment"":""Cadastro de clientes da Telefonica - contém informações de assinantes e segmentação""})",Map()
0,2025-12-15T23:28:09.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)","Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000, numOutputBytes -> 6119)"


In [0]:
%sql
-- Consultar a versão original da tabela (versão 0)
SELECT 
  'Versão Original (v0)' as versao,
  COUNT(*) as total_clientes,
  COUNT(DISTINCT segmento) as segmentos_distintos
FROM clientes VERSION AS OF 0

versao,total_clientes,segmentos_distintos
Versão Original (v0),1000,3


In [0]:
# Comparar diferentes versões da tabela
print("🔍 Comparando versões da tabela de clientes\n")
print("="*70)

# Obter número de versões
history = spark.sql("DESCRIBE HISTORY clientes")
max_version = history.agg({"version": "max"}).collect()[0][0]

print(f"\n📊 Total de versões: {max_version + 1} (0 a {max_version})\n")

# Comparar versão inicial vs atual
v0_count = spark.sql("SELECT COUNT(*) as count FROM clientes VERSION AS OF 0").collect()[0][0]
v_current_count = spark.table("clientes").count()

print(f"Versão 0 (Original):  {v0_count} clientes")
print(f"Versão {max_version} (Atual):     {v_current_count} clientes")
print(f"Diferença:            {v_current_count - v0_count:+d} clientes")

print("\n" + "="*70)

# Mostrar clientes que foram promovidos para Premium
print("\n🌟 Clientes promovidos para Premium (comparando versões):\n")

query = """
SELECT 
  atual.cliente_id,
  atual.nome,
  original.segmento as segmento_original,
  atual.segmento as segmento_atual
FROM clientes atual
INNER JOIN clientes VERSION AS OF 0 original
  ON atual.cliente_id = original.cliente_id
WHERE atual.segmento = 'Premium' 
  AND original.segmento != 'Premium'
LIMIT 5
"""

display(spark.sql(query))

🔍 Comparando versões da tabela de clientes


📊 Total de versões: 8 (0 a 7)

Versão 0 (Original):  1000 clientes
Versão 7 (Atual):     1002 clientes
Diferença:            +2 clientes


🌟 Clientes promovidos para Premium (comparando versões):



cliente_id,nome,segmento_original,segmento_atual
CLI00002,Cliente 2,Residencial,Premium
CLI00003,Cliente 3,Empresarial,Premium


In [0]:
# Consultar dados em um momento específico no tempo
from datetime import datetime, timedelta

print("🕒 Consultando dados por timestamp\n")

# Obter timestamp da primeira versão
history_df = spark.sql("DESCRIBE HISTORY clientes")
first_timestamp = history_df.orderBy("version").first()["timestamp"]

print(f"Timestamp da primeira versão: {first_timestamp}")
print(f"\nConsultando dados como estavam naquele momento...\n")

# Consultar usando timestamp
query = f"""
SELECT 
  segmento,
  COUNT(*) as total_clientes
FROM clientes TIMESTAMP AS OF '{first_timestamp}'
GROUP BY segmento
ORDER BY total_clientes DESC
"""

display(spark.sql(query))

🕒 Consultando dados por timestamp

Timestamp da primeira versão: 2025-12-15 23:28:09

Consultando dados como estavam naquele momento...



segmento,total_clientes
Premium,352
Residencial,329
Empresarial,319


### 🔄 Restaurando Versões Anteriores

O Delta Lake permite restaurar facilmente versões anteriores dos dados:

```sql
-- Restaurar para uma versão específica
RESTORE TABLE clientes TO VERSION AS OF 0;

-- Restaurar para um timestamp específico
RESTORE TABLE clientes TO TIMESTAMP AS OF '2024-01-01 10:00:00';
```

⚠️ **Nota**: Não vamos executar o RESTORE neste workshop para manter as alterações, mas esta funcionalidade é extremamente útil para:

* Recuperação de desastres
* Correção de erros em pipelines
* Testes e validações
* Auditoria e compliance

### 🎯 Benefícios do Delta Lake Demonstrados

✅ **Versionamento Automático**: Cada operação cria uma nova versão

✅ **Time Travel**: Acesso a qualquer versão histórica dos dados

✅ **Auditoria Completa**: Histórico de todas as operações (INSERT, UPDATE, DELETE)

✅ **Recuperação Rápida**: RESTORE para voltar a versões anteriores

✅ **Consultas Flexíveis**: Query por versão ou timestamp

✅ **Performance**: Otimizações automáticas e compactação

---

💡 **Próximo Passo**: Vamos criar Metric Views para análise semântica dos dados!

## 5. Metric Views - Camada Semântica

**Metric Views** são objetos do Unity Catalog que centralizam definições de métricas de negócio.

### Por que usar Metric Views?

* **Consistência**: Métricas definidas uma única vez, usadas em todos os lugares
* **Reutilização**: Mesmas métricas em dashboards, notebooks e Genie Spaces
* **Flexibilidade**: Separação entre definição de métricas e agregações
* **Análise Semântica**: IA entende as métricas e pode responder perguntas em linguagem natural
* **Governança**: Controle centralizado de KPIs empresariais

### Estrutura de uma Metric View:

```yaml
version: 1.1
source: tabela_origem
dimensions:  # Atributos para agrupamento
  - name: Dimensão
    expr: expressão_sql
measures:    # Métricas agregadas
  - name: Métrica
    expr: função_agregada
```

In [0]:
%sql
-- Criar Metric View para análise de receita
CREATE OR REPLACE VIEW metricas_receita
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: receita_mensal
  comment: Métricas de receita e faturamento da Telefonica
  
  dimensions:
    - name: Mes Referencia
      expr: mes_referencia
      display_name: Mês de Referência
      comment: Mês de faturamento
      synonyms:
        - mes
        - periodo
        - data
    
    - name: Status Pagamento
      expr: status_pagamento
      display_name: Status do Pagamento
      comment: Situação do pagamento (Pago, Pendente, Atrasado)
      synonyms:
        - status
        - situacao
    
    - name: Tipo Plano
      expr: CASE 
              WHEN plano_id IN ('PLAN001', 'PLAN002') THEN 'Básico'
              WHEN plano_id IN ('PLAN003', 'PLAN004') THEN 'Intermediário'
              WHEN plano_id IN ('PLAN005') THEN 'Premium'
              ELSE 'Empresarial'
            END
      display_name: Tipo de Plano
      synonyms:
        - categoria plano
        - segmento plano
  
  measures:
    - name: Receita Total
      expr: SUM(valor_total)
      display_name: Receita Total
      comment: Soma de todos os valores faturados
      format:
        type: currency
        currency_code: BRL
      synonyms:
        - faturamento
        - receita
        - vendas
    
    - name: Receita Base
      expr: SUM(valor_base)
      display_name: Receita Base
      comment: Receita proveniente dos planos (sem excedentes)
      format:
        type: currency
        currency_code: BRL
    
    - name: Receita Excedente
      expr: SUM(valor_excedente)
      display_name: Receita de Excedentes
      comment: Receita adicional por uso acima da franquia
      format:
        type: currency
        currency_code: BRL
      synonyms:
        - excedente
        - adicional
    
    - name: Total Descontos
      expr: SUM(desconto)
      display_name: Total de Descontos
      comment: Soma de todos os descontos concedidos
      format:
        type: currency
        currency_code: BRL
    
    - name: Numero Faturas
      expr: COUNT(1)
      display_name: Número de Faturas
      comment: Quantidade total de faturas
      synonyms:
        - quantidade faturas
        - total faturas
    
    - name: Ticket Medio
      expr: SUM(valor_total) / COUNT(1)
      display_name: Ticket Médio
      comment: Valor médio por fatura
      format:
        type: currency
        currency_code: BRL
      synonyms:
        - valor medio
        - media fatura
    
    - name: Taxa Inadimplencia
      expr: SUM(CASE WHEN status_pagamento IN ('Pendente', 'Atrasado') THEN 1 ELSE 0 END) / COUNT(1) * 100
      display_name: Taxa de Inadimplência (%)
      comment: Percentual de faturas não pagas
      format:
        type: number
      synonyms:
        - inadimplencia
        - atraso
$$;

SELECT '✅ Metric View "metricas_receita" criada com sucesso!' as status

status
"✅ Metric View ""metricas_receita"" criada com sucesso!"


In [0]:
%sql
-- Criar Metric View para análise de uso
CREATE OR REPLACE VIEW metricas_uso
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: uso_diario
  comment: Métricas de consumo de dados, voz e SMS
  
  dimensions:
    - name: Data Uso
      expr: data_uso
      display_name: Data de Uso
      comment: Data do registro de consumo
      synonyms:
        - data
        - dia
    
    - name: Mes Uso
      expr: DATE_TRUNC('MONTH', data_uso)
      display_name: Mês de Uso
      comment: Mês do consumo
      synonyms:
        - mes
        - periodo
    
    - name: Dia Semana
      expr: DAYOFWEEK(data_uso)
      display_name: Dia da Semana
      comment: Dia da semana (1=Domingo, 7=Sábado)
  
  measures:
    - name: Consumo Total GB
      expr: SUM(consumo_gb)
      display_name: Consumo Total (GB)
      comment: Volume total de dados consumidos em Gigabytes
      format:
        type: number
      synonyms:
        - dados consumidos
        - volume dados
        - trafego
    
    - name: Consumo Medio GB
      expr: AVG(consumo_gb)
      display_name: Consumo Médio (GB)
      comment: Média de consumo diário por cliente
      format:
        type: number
      synonyms:
        - media consumo
    
    - name: Total Minutos Voz
      expr: SUM(minutos_voz)
      display_name: Total de Minutos de Voz
      comment: Soma de todos os minutos de ligações
      format:
        type: number
      synonyms:
        - minutos
        - ligacoes
        - voz
    
    - name: Total SMS
      expr: SUM(sms_enviados)
      display_name: Total de SMS Enviados
      comment: Quantidade total de mensagens SMS
      format:
        type: number
      synonyms:
        - mensagens
        - sms
    
    - name: Clientes Ativos
      expr: COUNT(DISTINCT cliente_id)
      display_name: Clientes Ativos
      comment: Número de clientes únicos com uso registrado
      synonyms:
        - usuarios ativos
        - assinantes
    
    - name: Registros Uso
      expr: COUNT(1)
      display_name: Registros de Uso
      comment: Quantidade total de registros
$$;

SELECT '✅ Metric View "metricas_uso" criada com sucesso!' as status

status
"✅ Metric View ""metricas_uso"" criada com sucesso!"


In [0]:
%sql
-- Criar Metric View combinando clientes com uso e receita
CREATE OR REPLACE VIEW metricas_clientes
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: |
    SELECT 
      c.cliente_id,
      c.segmento,
      c.cidade,
      c.data_cadastro,
      DATEDIFF(CURRENT_DATE(), c.data_cadastro) as dias_cliente,
      u.data_uso,
      u.consumo_gb,
      u.minutos_voz,
      u.sms_enviados,
      r.valor_total,
      r.status_pagamento
    FROM clientes c
    LEFT JOIN uso_diario u ON c.cliente_id = u.cliente_id
    LEFT JOIN receita_mensal r ON c.cliente_id = r.cliente_id
  comment: Métricas consolidadas por cliente
  
  dimensions:
    - name: Segmento Cliente
      expr: segmento
      display_name: Segmento do Cliente
      comment: Segmentação (Residencial, Empresarial, Premium)
      synonyms:
        - segmento
        - categoria
    
    - name: Cidade
      expr: cidade
      display_name: Cidade
      comment: Cidade do cliente
      synonyms:
        - localidade
        - regiao
    
    - name: Faixa Tempo Cliente
      expr: CASE 
              WHEN dias_cliente < 90 THEN 'Novo (< 3 meses)'
              WHEN dias_cliente < 365 THEN 'Recente (3-12 meses)'
              WHEN dias_cliente < 730 THEN 'Estabelecido (1-2 anos)'
              ELSE 'Fiel (> 2 anos)'
            END
      display_name: Tempo como Cliente
      synonyms:
        - antiguidade
        - tempo cadastro
  
  measures:
    - name: Total Clientes
      expr: COUNT(DISTINCT cliente_id)
      display_name: Total de Clientes
      comment: Número total de clientes únicos
      synonyms:
        - quantidade clientes
        - numero clientes
    
    - name: Consumo Medio por Cliente
      expr: SUM(consumo_gb) / COUNT(DISTINCT cliente_id)
      display_name: Consumo Médio por Cliente (GB)
      comment: Média de consumo de dados por cliente
      format:
        type: number
    
    - name: Receita por Cliente
      expr: SUM(valor_total) / COUNT(DISTINCT cliente_id)
      display_name: Receita Média por Cliente
      comment: ARPU - Average Revenue Per User
      format:
        type: currency
        currency_code: BRL
      synonyms:
        - arpu
        - receita media
        - valor medio cliente
$$;

SELECT '✅ Metric View "metricas_clientes" criada com sucesso!' as status

status
"✅ Metric View ""metricas_clientes"" criada com sucesso!"


In [0]:
%sql
-- Consultar Metric View de receita
-- IMPORTANTE: Sempre usar MEASURE() para métricas e GROUP BY ALL

SELECT 
  `Mes Referencia`,
  `Status Pagamento`,
  MEASURE(`Receita Total`) AS receita_total,
  MEASURE(`Numero Faturas`) AS num_faturas,
  MEASURE(`Ticket Medio`) AS ticket_medio,
  MEASURE(`Taxa Inadimplencia`) AS taxa_inadimplencia
FROM metricas_receita
GROUP BY ALL
ORDER BY `Mes Referencia` DESC, receita_total DESC

Mes Referencia,Status Pagamento,receita_total,num_faturas,ticket_medio,taxa_inadimplencia
2025-12,Pago,61703.300000000185,297,207.75521885521948,0.0
2025-12,Pendente,21088.780000000006,105,200.84552380952385,100.0
2025-12,Atrasado,16585.22999999999,98,169.2370408163264,100.0
2025-11,Pago,53564.92000000006,303,176.78191419141933,0.0
2025-11,Pendente,21253.929999999986,106,200.50877358490553,100.0
2025-11,Atrasado,17462.039999999994,91,191.89054945054937,100.0
2025-10,Pago,53658.54000000017,300,178.86180000000056,0.0
2025-10,Atrasado,18409.730000000003,105,175.33076190476194,100.0
2025-10,Pendente,17067.18999999999,95,179.6546315789473,100.0


In [0]:
%sql
-- Análise de consumo por mês

SELECT 
  `Mes Uso`,
  MEASURE(`Consumo Total GB`) AS consumo_total_gb,
  MEASURE(`Consumo Medio GB`) AS consumo_medio_gb,
  MEASURE(`Total Minutos Voz`) AS total_minutos,
  MEASURE(`Total SMS`) AS total_sms,
  MEASURE(`Clientes Ativos`) AS clientes_ativos
FROM metricas_uso
GROUP BY ALL
ORDER BY `Mes Uso` DESC

Mes Uso,consumo_total_gb,consumo_medio_gb,total_minutos,total_sms,clientes_ativos
2025-12-01T00:00:00.000Z,48911.939999999995,6.987419999999999,1070312,177620,500
2025-11-01T00:00:00.000Z,106179.43999999999,7.078629333333333,2326704,375367,500
2025-10-01T00:00:00.000Z,110170.36000000006,7.1077651612903265,2406299,386558,500
2025-09-01T00:00:00.000Z,53162.909999999996,7.088387999999999,1168048,188397,500


In [0]:
%sql
-- Análise por segmento de cliente

SELECT 
  `Segmento Cliente`,
  `Cidade`,
  MEASURE(`Total Clientes`) AS total_clientes,
  MEASURE(`Consumo Medio por Cliente`) AS consumo_medio_gb,
  MEASURE(`Receita por Cliente`) AS arpu
FROM metricas_clientes
GROUP BY ALL
ORDER BY arpu DESC

Segmento Cliente,Cidade,total_clientes,consumo_medio_gb,arpu
Premium,Belo Horizonte,68,1013.739705882353,29390.5985294118
Residencial,Brasília,65,1130.4512307692305,28664.543076923117
Residencial,Salvador,56,1211.3555357142852,28137.40714285717
Empresarial,Belo Horizonte,57,964.6652631578946,27597.60000000004
Empresarial,Rio de Janeiro,66,1149.2440909090906,26704.85454545458
Empresarial,São Paulo,75,1011.7748000000004,25543.104000000043
Residencial,São Paulo,69,913.0626086956523,25029.87391304351
Premium,Brasília,86,866.7920930232558,25017.415116279077
Empresarial,Brasília,59,1008.5832203389829,24004.70847457629
Residencial,Rio de Janeiro,65,777.5044615384612,23814.733846153864


In [0]:
%sql
-- Análise de composição da receita

SELECT 
  `Tipo Plano`,
  MEASURE(`Receita Total`) AS receita_total,
  MEASURE(`Receita Base`) AS receita_base,
  MEASURE(`Receita Excedente`) AS receita_excedente,
  MEASURE(`Total Descontos`) AS descontos,
  -- Calcular percentuais (aritmética após agregação)
  ROUND(MEASURE(`Receita Excedente`) / MEASURE(`Receita Total`) * 100, 2) AS perc_excedente,
  ROUND(MEASURE(`Total Descontos`) / MEASURE(`Receita Total`) * 100, 2) AS perc_desconto
FROM metricas_receita
GROUP BY ALL
ORDER BY receita_total DESC

Tipo Plano,receita_total,receita_base,receita_excedente,descontos,perc_excedente,perc_desconto
Empresarial,167387.8199999999,166258.2999999998,1868.05,738.5300000000002,1.12,0.44
Intermediário,45890.20000000007,44509.300000000076,2120.25,739.3500000000001,4.62,1.61
Premium,45487.54000000004,45177.40000000005,766.9000000000001,456.76,1.69,1.0
Básico,22028.099999999944,20954.999999999913,2109.67,1036.5699999999997,9.58,4.71


### 🎯 Metric Views Criadas

Criamos 3 Metric Views com métricas de negócio:

1. **metricas_receita**: Análise de faturamento e inadimplência
   * Receita Total, Base e Excedente
   * Ticket Médio
   * Taxa de Inadimplência

2. **metricas_uso**: Análise de consumo
   * Consumo de Dados (GB)
   * Minutos de Voz
   * SMS Enviados
   * Clientes Ativos

3. **metricas_clientes**: Análise por cliente
   * Segmentação
   * ARPU (Average Revenue Per User)
   * Consumo Médio

---

### 💡 Regras Importantes para Consultar Metric Views:

✅ Sempre usar `MEASURE()` para envolver métricas

✅ Sempre usar `GROUP BY ALL`

✅ Usar backticks (\`) para nomes com espaços

✅ Aritmética entre métricas DEPOIS da agregação

❌ Nunca usar `SELECT *`

❌ Nunca usar `MIN()` ou `MAX()` em métricas

## 6. Dashboards & Genie Spaces

Agora que temos nossas Metric Views criadas, podemos usá-las para:

* **Dashboards**: Visualizações interativas e relatórios
* **Genie Spaces**: Análise conversacional com IA

Ambos utilizam as mesmas métricas definidas, garantindo consistência!

### 📊 Criando um Dashboard

#### Passo a Passo:

1. **Acesse**: Menu lateral > **Dashboards** > **Create Dashboard**
2. **Adicione as Metric Views**: metricas_receita, metricas_uso, metricas_clientes
3. **Adicione visualizações** usando as queries abaixo

---

#### 📈 Sugestões de Visualizações:

**1. KPIs Principais (Counter Widgets)**
```sql
-- Receita Total do Mês
SELECT 
  MEASURE(`Receita Total`) AS receita_total
FROM metricas_receita
WHERE `Mes Referencia` = DATE_FORMAT(CURRENT_DATE(), 'yyyy-MM')
GROUP BY ALL
```

```sql
-- Total de Clientes Ativos
SELECT 
  MEASURE(`Clientes Ativos`) AS clientes_ativos
FROM metricas_uso
WHERE `Mes Uso` = DATE_TRUNC('MONTH', CURRENT_DATE())
GROUP BY ALL
```

```sql
-- Taxa de Inadimplência
SELECT 
  ROUND(MEASURE(`Taxa Inadimplencia`), 2) AS taxa_inadimplencia
FROM metricas_receita
WHERE `Mes Referencia` = DATE_FORMAT(CURRENT_DATE(), 'yyyy-MM')
GROUP BY ALL
```

**2. Evolução da Receita (Line Chart)**
```sql
SELECT 
  `Mes Referencia` AS mes,
  MEASURE(`Receita Total`) AS receita_total,
  MEASURE(`Receita Base`) AS receita_base,
  MEASURE(`Receita Excedente`) AS receita_excedente
FROM metricas_receita
GROUP BY ALL
ORDER BY mes
```

**3. Receita por Tipo de Plano (Bar Chart)**
```sql
SELECT 
  `Tipo Plano` AS tipo_plano,
  MEASURE(`Receita Total`) AS receita,
  MEASURE(`Numero Faturas`) AS num_faturas
FROM metricas_receita
GROUP BY ALL
ORDER BY receita DESC
```

**4. Distribuição de Clientes por Segmento (Pie Chart)**
```sql
SELECT 
  `Segmento Cliente` AS segmento,
  MEASURE(`Total Clientes`) AS total_clientes
FROM metricas_clientes
GROUP BY ALL
ORDER BY total_clientes DESC
```

**5. Consumo de Dados por Mês (Area Chart)**
```sql
SELECT 
  `Mes Uso` AS mes,
  MEASURE(`Consumo Total GB`) AS consumo_gb,
  MEASURE(`Clientes Ativos`) AS clientes
FROM metricas_uso
GROUP BY ALL
ORDER BY mes
```

**6. Top Cidades por Receita (Table)**
```sql
SELECT 
  `Cidade` AS cidade,
  MEASURE(`Total Clientes`) AS clientes,
  MEASURE(`Receita por Cliente`) AS arpu,
  MEASURE(`Consumo Medio por Cliente`) AS consumo_medio_gb
FROM metricas_clientes
GROUP BY ALL
ORDER BY arpu DESC
LIMIT 10
```

**7. Análise de Inadimplência (Table)**
```sql
SELECT 
  `Mes Referencia` AS mes,
  `Status Pagamento` AS status,
  MEASURE(`Numero Faturas`) AS num_faturas,
  MEASURE(`Receita Total`) AS receita,
  MEASURE(`Taxa Inadimplencia`) AS taxa_inadimplencia
FROM metricas_receita
GROUP BY ALL
ORDER BY mes DESC, receita DESC
```

**8. Performance por Segmento (Table)**
```sql
SELECT 
  `Segmento Cliente` AS segmento,
  `Faixa Tempo Cliente` AS tempo_cliente,
  MEASURE(`Total Clientes`) AS clientes,
  MEASURE(`Receita por Cliente`) AS arpu,
  MEASURE(`Consumo Medio por Cliente`) AS consumo_gb
FROM metricas_clientes
GROUP BY ALL
ORDER BY arpu DESC
```

### 🧞 Criando um Genie Space

**Genie** é o assistente de IA da Databricks que permite análise conversacional de dados.

#### Passo a Passo:

1. **Acesse**: Menu lateral > **Genie Spaces** > **Create Genie Space**

2. **Configure**:
   * **Nome**: "Análise Telefonica"
   * **Descrição**: "Análise de receita, uso e clientes da Telefonica"

3. **Adicione as Metric Views**:
   * `metricas_receita`
   * `metricas_uso`
   * `metricas_clientes`

4. **Adicione Instruções** (opcional):
   ```
   Este espaço contém dados de telecomunicações da Telefonica.
   
   Métricas disponíveis:
   - Receita: faturamento, ticket médio, inadimplência
   - Uso: consumo de dados, minutos de voz, SMS
   - Clientes: segmentação, ARPU, distribuição geográfica
   
   Valores monetários em BRL (Reais).
   ```


### 💬 Exemplos de Perguntas para o Genie

Após criar o Genie Space, você pode fazer perguntas em linguagem natural:

#### Perguntas sobre Receita:
* "Qual foi a receita total no último mês?"
* "Mostre a evolução da receita nos últimos 3 meses"
* "Qual tipo de plano gera mais receita?"
* "Qual é a taxa de inadimplência atual?"
* "Compare receita base vs receita de excedentes"

#### Perguntas sobre Uso:
* "Quantos clientes ativos temos?"
* "Qual é o consumo médio de dados por cliente?"
* "Mostre o consumo total de dados por mês"
* "Quantos minutos de voz foram usados este mês?"

#### Perguntas sobre Clientes:
* "Quantos clientes temos por segmento?"
* "Qual cidade tem o maior ARPU?"
* "Mostre a distribuição de clientes por tempo de cadastro"
* "Qual segmento consome mais dados?"

#### Perguntas Complexas:
* "Qual é a correlação entre consumo de dados e receita?"
* "Compare o ARPU entre clientes novos e antigos"
* "Identifique tendências de crescimento na receita"
* "Quais segmentos têm maior taxa de inadimplência?"

---

💡 **O Genie entende sinônimos** definidos nas Metric Views e pode responder em linguagem natural!

## 🎉 Parabéns! Workshop Concluído

### O que aprendemos:

✅ **Unity Catalog**
* Estrutura hierárquica (Catalog > Schema > Tables)
* Metadados enriquecidos e documentação
* Governança e controle de acesso
* Information Schema para consultas de metadados

✅ **Delta Lake**
* Formato ACID para data lakes
* Time Travel para consultas históricas
* Versionamento automático
* Auditoria completa de operações

✅ **Metric Views**
* Camada semântica para métricas de negócio
* Definição centralizada de KPIs
* Reutilização em dashboards e Genie
* Sinônimos para descoberta por IA

✅ **Dashboards & Genie**
* Visualizações interativas
* Análise conversacional com IA
* Consistência de métricas

---

### 🚀 Próximos Passos:

1. **Explore o Data Explorer**: Visualize linhagem e permissões
2. **Crie seu Dashboard**: Use as queries fornecidas
3. **Configure um Genie Space**: Teste perguntas em linguagem natural
4. **Experimente com seus dados**: Adapte este notebook para casos reais
5. **Compartilhe**: Colabore com sua equipe usando Unity Catalog

---

### 📚 Recursos Adicionais:

* [Documentação Unity Catalog](https://docs.databricks.com/data-governance/unity-catalog/index.html)
* [Documentação Delta Lake](https://docs.databricks.com/delta/index.html)
* [Documentação Metric Views](https://docs.databricks.com/sql/language-manual/sql-ref-metric-views.html)
* [Documentação Genie](https://docs.databricks.com/genie/index.html)

---

👋 **Obrigado por participar do Workshop Databricks - Telefonica!**

## 🧹 Limpeza (Opcional)

Se desejar remover os recursos criados neste workshop:

```python
# ATENÇÃO: Isso irá deletar TODOS os dados criados!
# Descomente as linhas abaixo apenas se tiver certeza

# spark.sql(f"DROP SCHEMA IF EXISTS {catalog_name}.{schema_name} CASCADE")
# spark.sql(f"DROP CATALOG IF EXISTS {catalog_name} CASCADE")
# print(f"✅ Catálogo {catalog_name} removido com sucesso!")
```

⚠️ **Importante**: 
* Esta ação é **irreversível**
* Todos os dados, tabelas e metric views serão deletados
* Use apenas em ambientes de teste/desenvolvimento